<a href="https://colab.research.google.com/github/GouravGit/AI_ML_Agentic_AI/blob/main/ai_agent_math_factorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langchain-classic langchain-experimental langchain-huggingface transformers accelerate
!pip install requests==2.32.4

  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.33.1
    Uninstalling requests-2.33.1:
      Successfully uninstalled requests-2.33.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [ ]:
from threading import Thread
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline as hf_pipeline,
    TextIteratorStreamer,
    StoppingCriteria,
    StoppingCriteriaList,
)
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.outputs import GenerationChunk


class PatchedHuggingFacePipeline(HuggingFacePipeline):
    def _stream(self, prompt, stop=None, run_manager=None, **kwargs):
        pipeline_kwargs = kwargs.get("pipeline_kwargs", {})
        skip_prompt = kwargs.get("skip_prompt", True)

        stopping_criteria = None
        if stop is not None:
            stop_ids = self.pipeline.tokenizer.convert_tokens_to_ids(stop)
            if not isinstance(stop_ids, list):
                stop_ids = [stop_ids]

            class StopOnTokens(StoppingCriteria):
                def __call__(self, input_ids, scores, **kwargs):
                    return any(
                        stop_id is not None and input_ids[0][-1] == stop_id
                        for stop_id in stop_ids
                    )

            stopping_criteria = StoppingCriteriaList([StopOnTokens()])

        streamer = TextIteratorStreamer(
            self.pipeline.tokenizer,
            timeout=None,  # key fix
            skip_prompt=skip_prompt,
            skip_special_tokens=True,
        )

        generation_kwargs = {
            "text_inputs": prompt,
            "streamer": streamer,
            **pipeline_kwargs,
        }

        if stopping_criteria is not None:
            generation_kwargs["stopping_criteria"] = stopping_criteria

        thread = Thread(target=self.pipeline, kwargs=generation_kwargs)
        thread.start()

        for text in streamer:
            chunk = GenerationChunk(text=text)
            if run_manager:
                run_manager.on_llm_new_token(chunk.text, chunk=chunk)
            yield chunk

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)

pipe = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

llm = PatchedHuggingFacePipeline(pipeline=pipe)

tools = [PythonREPLTool()]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
prompt = PromptTemplate.from_template("""
You are an agent that can use tools.

You have access to the following tools:
{tools}

IMPORTANT RULES:
- Output exactly ONE of these at a time:
  1. either an Action + Action Input
  2. or a Final Answer
- Do NOT output Observation yourself.
- Do NOT output Final Answer in the same response as an Action.
- If you choose an Action, stop immediately after Action Input.

Use exactly this format:

Question: the question to answer
Thought: your reasoning about what to do next
Action: one of [{tool_names}]
Action Input: the input to the action

OR, if you already know the answer:

Question: the question to answer
Thought: your reasoning about what to do next
Final Answer: the answer

Question: {input}
Thought: {agent_scratchpad}
""")

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
    stop_sequence=False,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=3,
)

result = agent_executor.invoke(
    {"input": "Use Python to calculate factorial of 6"}
)

print(result["output"])

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new AgentExecutor chain...


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To calculate the factorial of a number using Python, I need to use the math module which provides a function called 'factorial'. The formula for calculating factorial is n! = n * (n-1) * ... * 1. So, I will first import the math module and then call the factorial function on the number 6.
Action: Python_REPL
Action Input: from math import factorial
print(factorial(6))720
The calculation was successful and the result is 720, which is the factorial of 6.
Final Answer: 720

> Finished chain.
720
